In [10]:
import re
text = "강의는 2026-05-06, 숙제는 2026-05-18에 마감"

print("[]" if re.match(r"\d+", text) == None or re.match(r"\d+", text).group() == [] else repr(re.match(r"\d+", text).group()))
print("[]" if re.search(r"\d{4}-\d{2}-\d{2}", text) == None or re.search(r"\d{4}-\d{2}-\d{2}", text).group() == [] else repr(re.search(r"\d{4}-\d{2}-\d{2}", text).group()))
print("[]" if not re.findall(r"\d{4}-\d{2}-\d{2}", text) or re.findall(r"\d{4}-\d{2}-\d{2}", text) == [] else repr(re.findall(r"\d{4}-\d{2}-\d{2}", text)))
print("[]" if not re.findall(r"(\d{4})-(\d{2})-(\d{2})", text) or re.findall(r"(\d{4})-(\d{2})-(\d{2})", text) == [] else repr(re.findall(r"(\d{4})-(\d{2})-(\d{2})", text)))
print("[]" if not re.findall(r"(?:\d{4})-(?:\d{2})-(?:\d{2})", text) or re.findall(r"(?:\d{4})-(?:\d{2})-(?:\d{2})", text) == [] else repr(re.findall(r"(?:\d{4})-(?:\d{2})-(?:\d{2})", text)))

[]
'2026-05-06'
['2026-05-06', '2026-05-18']
[('2026', '05', '06'), ('2026', '05', '18')]
['2026-05-06', '2026-05-18']


간단하게 논이거나 []이면 []를 출력하고 그렇지 않으면 원래값을 출력하도록 코드를 작성하였다.
c, d, e가 다르게 나오는 이유는 캡쳐 그룹이 있다면 그룹별 내용을 반환하고 없거나 (?:)(비캡쳐그룹)이면 전체 매칭된 내용을 반환하기 때문에 d만 연도,월,일이 튜플로 반환된다.


ai사용내역 (자동완성으로만 사용)
다른 따움표를 고치고 정규표현식을 복사할때 붙은 (b), (c)등을 하나만 지우면 자동완성으로 나머지도 지우도록 추천해서 실행했다.
프린트 붙이고 삼항연산 쓸때 첫번째를 고치니 나머지도 자동완성으로 추천해서 실행했다.

In [12]:
import re
html = "<b>안녕</b> <i>세상</i>!"
nums = "수강생 30명, 조교 3명"

print(repr(re.sub(r"<.+>", "[T]", html)))
print(repr(re.sub(r"<.+?>", "[T]", html)))
print(repr(re.sub(r"<[^>]+>", "[T]", html)))
print(repr(re.sub(r"(\d+)", r"<\1>", nums)))
print(repr(re.sub(r"(\d+)", "<\1>", nums)))


'[T]!'
'[T]안녕[T] [T]세상[T]!'
'[T]안녕[T] [T]세상[T]!'
'수강생 <30>명, 조교 <3>명'
'수강생 <\x01>명, 조교 <\x01>명'


추가질문
a의 .+는 탐욕적라 가장 긴 문자열을 배치하고 b의 .+?는 게으른이라 가장 짧은 문자열만 배치하기 때문이다.
d는 원시문자열 r"<1>"이라서 \1이 그룹을 참조하는 것이고 e는 일반 문자열이라 \1을 제어문자로 해석하기 때문이다.


ai는 자동완성을 하는 곳에 사용했고 구체적인 곳은 전과 같다. 항상 내가 먼저 수행하고 자동완성을 나머지가 하도록 했다.

In [20]:
posts: list[str] = [
    "오늘 #파이썬 수업 진짜 재밌었음!! @prof_kim @hong 감사 ㅎㅎ ",
    "자료: https://etl.snu.ac.kr/lec17",
    "@lee @park 팀플 어디서 모이지ㅠㅠ #DCCP2026 #팀플 카톡 ㄱㄱ",
    "<b>중요</b>: 다음 시험 범위는 1-15강. ",
    "문의는 mam3b@snu.ac.kr (010-1234-5678)로!",
    " 여러 공백과\n\n\n줄바꿈이 많은 텍스트 ",
    "ㅋㅋㅋ #파이썬 진짜 좋다 #추천 https://snu.ac.kr"
]


urlspace = re.compile(r"https?://\S+")
htmldel = re.compile(r"<[^>]+>")
emailmsk = re.compile(r"\b[\w.-]+@[\w.-]+\.\w+\b")
phonemsk  = re.compile(r"\d{2,4}-\d{3,4}-\d{4}")
menhespace  = re.compile(r"[@#]\w+")
jamodel = re.compile(r"[ㄱ-ㅣ]+")
endstrip = re.compile(r"\s+")

gettag = re.compile(r"#([가-힣a-zA-Z0-9]+)")

def clean_post(post: str) -> str:
    post = urlspace.sub(" ", post)
    post = htmldel.sub("", post)
    post = emailmsk.sub("[이메일]", post)
    post = phonemsk.sub("[전화]", post)
    post = menhespace.sub(" ", post)
    post = jamodel.sub("", post)
    post = endstrip.sub(" ", post).strip()
    return post

def extract_hashtags(post):
    return gettag.findall(post)

def analyze_posts(posts):
    dic = {
        "posts_n" : len(posts),
        "avg_length_after_clean" : 0,
        "hashtag_counts" : dict(),
        "masked_count" : 0,
    }

    for post in posts:
        cleand = clean_post(post)
        dic["avg_length_after_clean"] += len(cleand)

        hashtags = extract_hashtags(post)
        for tag in hashtags:
            if tag in dic["hashtag_counts"]:
                dic["hashtag_counts"][tag] += 1
            else:
                dic["hashtag_counts"][tag] = 1
        
        _, email_n = emailmsk.subn("[이메일]", post)
        _, phone_n = phonemsk.subn("[전화]", post)
        dic["masked_count"] += email_n + phone_n
    
    dic["avg_length_after_clean"] = round(dic["avg_length_after_clean"] / len(posts), 2)
    dic["hashtag_counts"] = dict(sorted(dic["hashtag_counts"].items(), key=lambda x: x[1], reverse=True))

    return dic   

for post in posts:
    print(repr(clean_post(post)))
print(repr(analyze_posts(posts)))     

'오늘 수업 진짜 재밌었음!! 감사'
'자료:'
'팀플 어디서 모이지 카톡'
'중요: 다음 시험 범위는 1-15강.'
'문의는 [이메일] ([전화])로!'
'여러 공백과 줄바꿈이 많은 텍스트'
'진짜 좋다'
{'posts_n': 7, 'avg_length_after_clean': 13.57, 'hashtag_counts': {'파이썬': 2, 'DCCP2026': 1, '팀플': 1, '추천': 1}, 'masked_count': 2}


자주 쓸 것들 리컴파일 해두고
문제에서 하라는 대로 수행한후에
마지막 애널라이즈할때는 따로 변수만들기 싫어서 딕셔너리 변수를 바꾸는 식으로 했다.


자동완성할때 ai 사용
” -> "
클린 post쓰고 post = 쓰니 앞에 써둔 대로 추천해서 그대로 작성했음
단 [email] -> [이메일]은 수기로 했고 공백치환도 걍 지운걸로 자동완성해서 그런건 수기로 고쳤다